# Notebook 04 — Multi-Modal Integration

**Goal:** Join all three datasets into a single `master.csv` analytical foundation.

**Integration steps:**
1. Join marathon results + weather on `(city_clean, race_date)`
2. Join weather-enriched marathon data + Strava training data on `(gender, age_group, city_clean, year)`
3. Export `master.csv`


In [1]:
import pandas as pd
import numpy as np
import os

PROC_DIR = '../data/processed'

# Compact dtypes keep the 2.85M-row marathon table small in RAM
NEEDED = {'gender', 'age', 'finish_min', 'year', 'race', 'city_clean', 'race_date'}
marathon = pd.read_csv(
    os.path.join(PROC_DIR, 'marathon_clean.csv'),
    usecols=lambda c: c in NEEDED,
    dtype={'gender': 'category', 'age': 'float32', 'finish_min': 'float32',
           'race': 'category', 'city_clean': 'category', 'race_date': 'category'},
    low_memory=False,  # single-pass read: chunked reads break category dtypes on sparse columns
)
athletes = pd.read_csv(os.path.join(PROC_DIR, 'athletes_clean.csv'))
weather  = pd.read_csv(os.path.join(PROC_DIR, 'weather.csv'))

print(f'Marathon:  {marathon.shape} — {marathon.memory_usage(deep=True).sum()/1e6:.0f} MB in RAM')
print(f'Athletes:  {athletes.shape}')
print(f'Weather:   {weather.shape}')

Marathon:  (2854273, 7) — 57 MB in RAM
Athletes:  (36412, 6)
Weather:   (89, 7)


## Step 1 — Join Marathon Results with Weather

Key: `(city_clean, race_date)` in marathon ↔ `(city, race_date)` in weather.

In [2]:
city_col = 'city_clean' if 'city_clean' in marathon.columns else 'city'

marathon_w = marathon.merge(
    weather.rename(columns={'city': city_col}),
    on=[city_col, 'race_date'],
    how='left'
)

n_matched = marathon_w['temp_mean_c'].notna().sum()
print(f'Rows with weather data: {n_matched:,} / {len(marathon_w):,}  ({n_matched/len(marathon_w)*100:.1f}%)')

Rows with weather data: 1,740,722 / 2,854,273  (61.0%)


## Step 2 — Join with Strava Training Data

The Strava dataset (`athletes_clean.csv`) contains 36,412 athletes with:
- Monthly running distance → converted to weekly km (distance / 4.33)
- Nationality (`country`)
- `major` field: comma-separated list of target marathons (e.g. `"CHICAGO 2019, NEW YORK 2018"`)

**Integration key:** For each athlete, extract `(city, year)` pairs from the `major` field, then aggregate median weekly km by `(gender, age_group, city, year)`. Join onto marathon runners on those same four keys.

In [3]:
import re

RACE_NAME_MAP = {
    'CHICAGO': 'Chicago', 'NEW YORK': 'New York City', 'HOUSTON': 'Houston',
    'LOS ANGELES': 'Los Angeles', 'TWIN CITIES': 'Minneapolis',
    'MARINE CORPS': 'Washington', 'HONOLULU': 'Honolulu',
    'PHILADELPHIA': 'Philadelphia', 'CALIFORNIA': 'Sacramento',
}

def extract_races(major_str):
    """Return list of (city, year) from a major string like 'CHICAGO 2019,NEW YORK 2018'."""
    if pd.isna(major_str):
        return []
    races = []
    for part in major_str.split(','):
        for key, city in RACE_NAME_MAP.items():
            m = re.search(rf'{key}\s+(\d{{4}})', part.strip())
            if m:
                races.append((city, int(m.group(1))))
    return races

# Expand athletes → one row per (athlete, target race)
rows = []
for _, ath in athletes.iterrows():
    for city, year in extract_races(ath['major']):
        rows.append({
            'gender':    ath['gender'],
            'age_group': ath['age_group'],
            'city_clean': city,
            'year':      year,
            'weekly_km': ath['weekly_km'],
        })

strava_expanded = pd.DataFrame(rows)
print(f'Strava race entries: {len(strava_expanded):,}')

# Aggregate to group medians
# Map Strava age groups (18-34, 35-54, 55+) to the same buckets used in master
AGE_MAP = {'18 - 34': '<30', '35 - 54': '30-39', '55 +': '60-69'}
strava_expanded['age_group'] = strava_expanded['age_group'].map(AGE_MAP)

strava_agg = (
    strava_expanded
    .groupby(['gender', 'age_group', 'city_clean', 'year'])['weekly_km']
    .median()
    .reset_index()
    .rename(columns={'weekly_km': 'strava_weekly_km'})
)
print(f'Aggregated groups: {len(strava_agg)}')

Strava race entries: 10,288
Aggregated groups: 104


In [4]:
# Bucket runner ages into the 3 Strava groups for the join key
# (<30 → '<30', 30–49 → '30-39', 50+ → '60-69'; matches AGE_MAP above)
marathon_w['age_group_strava'] = pd.cut(
    marathon_w['age'], bins=[0, 29, 49, 120],
    labels=['<30', '30-39', '60-69'], right=True
).astype(object)
marathon_w['year_int'] = marathon_w['year'].astype('Int64').astype(object)

master = marathon_w.merge(
    strava_agg.rename(columns={'age_group': 'age_group_strava', 'year': 'year_int'}),
    on=['gender', 'age_group_strava', 'city_clean', 'year_int'],
    how='left'
)

matched = master['strava_weekly_km'].notna().sum()
print(f'Runners with Strava weekly_km: {matched:,} / {len(master):,}  ({matched/len(master)*100:.1f}%)')

Runners with Strava weekly_km: 703,689 / 2,854,273  (24.7%)


In [5]:
# Add pace columns
if 'finish_min' in master.columns:
    master['pace_min_per_km'] = master['finish_min'] / 42.195

# Add age group
if 'age' in master.columns:
    bins   = [0, 29, 39, 49, 59, 69, 120]
    labels = ['<30', '30–39', '40–49', '50–59', '60–69', '70+']
    master['age_group'] = pd.cut(master['age'], bins=bins, labels=labels, right=True)

print('Master columns:', list(master.columns))
print(f'Master shape:   {master.shape}')

Master columns: ['gender', 'age', 'finish_min', 'year', 'race', 'city_clean', 'race_date', 'temp_mean_c', 'temp_max_c', 'precip_mm', 'wind_kmh', 'humidity_pct', 'age_group_strava', 'year_int', 'strava_weekly_km', 'pace_min_per_km', 'age_group']
Master shape:   (2854273, 17)


## Step 4 — Correlation Overview

In [6]:
numeric_cols = ['finish_min', 'age', 'strava_weekly_km',
                'temp_mean_c', 'temp_max_c', 'precip_mm', 'wind_kmh']
available = [c for c in numeric_cols if c in master.columns]
print('Correlation with finish_min:')
corr = master[available].corr()['finish_min'].drop('finish_min').sort_values()
print(corr.round(3))


Correlation with finish_min:


precip_mm          -0.046
wind_kmh            0.002
strava_weekly_km    0.042
temp_max_c          0.061
temp_mean_c         0.063
age                 0.085
Name: finish_min, dtype: float64


## Step 5 — Save Master DataFrame

In [7]:
master_out = os.path.join(PROC_DIR, 'master.csv')
master.to_csv(master_out, index=False)
print(f'Saved {master_out} — {len(master):,} rows × {master.shape[1]} columns')
print('\nColumn list:')
for c in master.columns:
    print(f'  {c}: {master[c].dtype}')


Saved ../data/processed/master.csv — 2,854,273 rows × 17 columns

Column list:
  gender: str
  age: float32
  finish_min: float32
  year: int64
  race: category
  city_clean: str
  race_date: str
  temp_mean_c: float64
  temp_max_c: float64
  precip_mm: float64
  wind_kmh: float64
  humidity_pct: float64
  age_group_strava: object
  year_int: object
  strava_weekly_km: float64
  pace_min_per_km: float32
  age_group: category
